In [0]:

# ============================================================
# 07_HIERARCHY_CONSTRUCTION
# Firmable - Employee Hierarchy Construction
#
# Inputs:
#   employees_clean
#   manager_assignments
#
# Outputs:
#   hierarchy_edges
#   employee_hierarchy
#
# Ground truth is never used.
# ============================================================

from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

EMPLOYEE_TABLE = "workspace.firmable_v2_silver.employees_clean"
ASSIGNMENT_TABLE = "workspace.firmable_v2_silver.manager_assignments"

EDGE_TABLE = "workspace.firmable_v2_silver.hierarchy_edges"
HIERARCHY_TABLE = "workspace.firmable_v2_silver.employee_hierarchy"

MAX_DEPTH = 20

# ============================================================
# 1. LOAD EMPLOYEE AND ASSIGNMENT DATA
# ============================================================

employees = spark.table(EMPLOYEE_TABLE).select(
    "company_id",
    "employee_id",
    "company_name",
    "job_title_clean",
    "department",
    "business_unit",
    "team_id",
    "team_name",
    "inferred_seniority_level",
    "management_authority",
    "title_function"
).dropDuplicates(["company_id", "employee_id"])

assignments = spark.table(ASSIGNMENT_TABLE).select(
    "company_id",
    "employee_id",
    F.col("candidate_manager_id").alias("manager_id"),
    "candidate_score",
    "relationship_type",
    "assignment_status",
    "assignment_round"
)

# ============================================================
# 2. BUILD EMPLOYEE-MANAGER BASE DATAFRAME
# ============================================================

base = (
    employees
    .join(
        assignments,
        ["company_id", "employee_id"],
        "left"
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

total_employees = base.count()

assert total_employees == employees.count(), (
    "Employee count changed after joining assignments."
)

# ============================================================
# 3. BUILD HIERARCHY EDGES
# ============================================================

edges = (
    base
    .filter(F.col("manager_id").isNotNull())
    .select(
        "company_id",
        "manager_id",
        "employee_id",
        "candidate_score",
        "relationship_type",
        "assignment_round"
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

manager_ids = (
    employees
    .select(
        "company_id",
        F.col("employee_id").alias("manager_id")
    )
    .dropDuplicates()
)

invalid_managers = (
    edges
    .join(
        F.broadcast(manager_ids),
        ["company_id", "manager_id"],
        "left_anti"
    )
    .count()
)

assert invalid_managers == 0, (
    f"Found {invalid_managers} invalid manager references."
)

# Every employee must have at most one manager.
duplicate_reports = (
    edges
    .groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_reports == 0, (
    f"Found {duplicate_reports} employees with multiple managers."
)

# ============================================================
# 4. IDENTIFY ROOT EMPLOYEES
# Employees without assigned managers are component roots.
# ============================================================

visited = (
    base
    .filter(F.col("manager_id").isNull())
    .select(
        "company_id",
        "employee_id",
        F.col("employee_id").alias("root_employee_id"),
        F.lit(0).alias("hierarchy_depth")
    )
    .dropDuplicates()
    .persist(StorageLevel.MEMORY_AND_DISK)
)

frontier = visited

root_count = visited.count()

assert root_count > 0, (
    "No hierarchy roots found. Check manager assignments."
)

# ============================================================
# 5. TRAVERSE HIERARCHY FROM ROOTS
# ============================================================

reached_max_depth = False

for depth in range(1, MAX_DEPTH + 1):

    parent_frontier = frontier.select(
        "company_id",
        F.col("employee_id").alias("parent_id"),
        "root_employee_id",
        "hierarchy_depth"
    )

    children = (
        edges.alias("e")
        .join(
            parent_frontier.alias("p"),
            (
                F.col("e.company_id") == F.col("p.company_id")
            ) & (
                F.col("e.manager_id") == F.col("p.parent_id")
            ),
            "inner"
        )
        .select(
            F.col("e.company_id").alias("company_id"),
            F.col("e.employee_id").alias("employee_id"),
            F.col("p.root_employee_id").alias("root_employee_id"),
            (
                F.col("p.hierarchy_depth") + 1
            ).alias("hierarchy_depth")
        )
        .join(
            visited.select("company_id", "employee_id"),
            ["company_id", "employee_id"],
            "left_anti"
        )
        .dropDuplicates()
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    new_children_count = children.count()

    print(
        f"Hierarchy depth {depth}: "
        f"{new_children_count} new employees reached"
    )

    if new_children_count == 0:
        children.unpersist()
        break

    previous_visited = visited

    visited = (
        visited
        .unionByName(children)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    # Materialize the updated visited DataFrame before releasing
    # previous cached data.
    visited.count()

    previous_visited.unpersist()

    frontier.unpersist()

    frontier = children

    if depth == MAX_DEPTH:
        reached_max_depth = True

# ============================================================
# 6. VERIFY THAT ALL EMPLOYEES ARE REACHABLE
# ============================================================

visited_count = visited.count()

unreachable_employees = (
    base
    .select("company_id", "employee_id")
    .join(
        visited.select("company_id", "employee_id"),
        ["company_id", "employee_id"],
        "left_anti"
    )
)

unreachable_count = unreachable_employees.count()

if unreachable_count > 0:
    raise Exception(
        f"{unreachable_count} employees are unreachable from roots. "
        "Possible cycle or hierarchy depth issue."
    )

if reached_max_depth:
    raise Exception(
        f"Hierarchy traversal reached MAX_DEPTH={MAX_DEPTH}. "
        "Increase MAX_DEPTH only after investigating the hierarchy."
    )

assert visited_count == total_employees, (
    f"Visited {visited_count} of {total_employees} employees."
)

# ============================================================
# 7. ATTACH HIERARCHY INFORMATION TO ALL EMPLOYEES
# ============================================================

hierarchy = (
    base
    .join(
        visited,
        ["company_id", "employee_id"],
        "left"
    )
    .withColumn(
        "is_root",
        F.col("manager_id").isNull()
    )
    .withColumn(
        "root_type",
        F.when(
            F.col("is_root") &
            (F.col("inferred_seniority_level") == 0),
            F.lit("CEO_ROOT")
        )
        .when(
            F.col("is_root"),
            F.lit("UNASSIGNED_ROOT")
        )
        .otherwise(F.lit("CHILD"))
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# ============================================================
# 8. HIERARCHY INTEGRITY CHECKS
# ============================================================

duplicate_rows = (
    hierarchy
    .groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_rows == 0, (
    f"Found {duplicate_rows} duplicate hierarchy rows."
)

invalid_depths = hierarchy.filter(
    F.col("hierarchy_depth").isNull() |
    (F.col("hierarchy_depth") < 0)
).count()

assert invalid_depths == 0, (
    f"Found {invalid_depths} invalid hierarchy depths."
)

# Validate manager seniority is strictly higher than employee seniority.
employee_levels = employees.select(
    "company_id",
    F.col("employee_id").alias("checked_employee_id"),
    F.col("inferred_seniority_level").alias("employee_level")
)

manager_levels = employees.select(
    "company_id",
    F.col("employee_id").alias("checked_manager_id"),
    F.col("inferred_seniority_level").alias("manager_level")
)

seniority_check = (
    hierarchy
    .filter(F.col("manager_id").isNotNull())
    .join(
        employee_levels,
        (
            hierarchy.company_id == employee_levels.company_id
        ) & (
            hierarchy.employee_id == employee_levels.checked_employee_id
        ),
        "inner"
    )
    .join(
        manager_levels,
        (
            hierarchy.company_id == manager_levels.company_id
        ) & (
            hierarchy.manager_id == manager_levels.checked_manager_id
        ),
        "inner"
    )
)

invalid_seniority = seniority_check.filter(
    F.col("manager_level") >= F.col("employee_level")
).count()

assert invalid_seniority == 0, (
    f"Found {invalid_seniority} invalid seniority relationships."
)

# ============================================================
# 9. PUBLISH HIERARCHY EDGES
# ============================================================

(
    edges.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(EDGE_TABLE)
)

# ============================================================
# 10. PUBLISH EMPLOYEE HIERARCHY
# ============================================================

(
    hierarchy.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(HIERARCHY_TABLE)
)

# ============================================================
# 11. FINAL SUMMARY
# ============================================================

final_hierarchy = spark.table(HIERARCHY_TABLE)

final_total = final_hierarchy.count()

assigned_count = final_hierarchy.filter(
    F.col("manager_id").isNotNull()
).count()

component_roots = final_hierarchy.filter(
    F.col("is_root")
).count()

maximum_depth = final_hierarchy.agg(
    F.max("hierarchy_depth")
).first()[0]

print("\n========== HIERARCHY CONSTRUCTION COMPLETED ==========")

print("Total employees:", final_total)
print("Assigned employees:", assigned_count)
print("Unassigned employees:", final_total - assigned_count)
print("Component roots:", component_roots)
print("Maximum hierarchy depth:", maximum_depth)
print("Unreachable employees:", unreachable_count)
print("Duplicate hierarchy rows:", duplicate_rows)
print("Invalid manager references:", invalid_managers)
print("Multiple-manager employees:", duplicate_reports)
print("Invalid seniority relationships:", invalid_seniority)

print("\n========== ROOT TYPE DISTRIBUTION ==========")

final_hierarchy.filter(
    F.col("is_root")
).groupBy(
    "root_type"
).count().orderBy(
    "root_type"
).show(truncate=False)

print("\n========== HIERARCHY DEPTH DISTRIBUTION ==========")

final_hierarchy.groupBy(
    "hierarchy_depth"
).count().orderBy(
    "hierarchy_depth"
).show(truncate=False)

print("\nOUTPUT TABLES:")
print(EDGE_TABLE)
print(HIERARCHY_TABLE)

print("\nALL HIERARCHY VALIDATIONS PASSED")

# ============================================================
# 12. RELEASE CACHED DATAFRAMES
# ============================================================

base.unpersist()
edges.unpersist()
visited.unpersist()
frontier.unpersist()
hierarchy.unpersist()
